# Download Market Data — CryptoStatArb

**Builds the project's canonical dataset.** Run once (or with `FORCE_REFRESH=True` for fresh data);
every other notebook just loads the pickle it writes.

### Universe construction
1. **Both exchanges:** keep coins listed on **both Binance.US and Coinbase** (USD/USDT), so the
   set is liquid and cross-listed. Stablecoins excluded.
2. **Top by volume:** rank by Binance.US 24h quote volume, take the top 200 (there are ~140 on
   both, so effectively all of them).
3. **Download** hourly OHLCV from **2022** to now (UTC), pruning coins listed too late to qualify.
4. **Coverage filter:** drop any coin without **≥90%** of hourly bars over the full window.

### Outputs (`data/`, gitignored — regenerate by re-running)
| file | contents |
|---|---|
| `binance_us_hourly_ohlcv.pk` | full Open/High/Low/Close/Volume panel, columns `(Field, Ticker)` |
| `binance_us_hourly_close.pk` | close-only view |
| `universe.csv` | final list of kept symbols |

**Load in any notebook:**
```python
panel = pd.read_pickle('data/binance_us_hourly_ohlcv.pk')
close, volume = panel['Close'], panel['Volume']
```

In [1]:
import time, json, ssl, certifi, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
from binance.client import Client

client = Client(tld='US')   # Binance.US public endpoints
SSL_CTX = ssl.create_default_context(cafile=certifi.where())   # macOS-safe cert bundle
print('python-binance ready')

python-binance ready


In [2]:
# --- Config ---
INTERVAL = '1h'
START = '2022-01-01'          # download window start (UTC)
TOP_N = 200                   # take top-N of the cross-listed set, ranked by Binance.US volume
MIN_COVERAGE = 0.90          # drop coins with < this fraction of hourly bars over the window
PRUNE_AFTER = pd.Timestamp('2022-06-01', tz='UTC')   # first bar later than this can't reach 90%
STABLES = {'USDT','USDC','DAI','TUSD','USDP','PYUSD','FDUSD','BUSD','GUSD','USDD',
           'FRAX','LUSD','EUR','GBP','USD','EURC','USTC'}

DATA_DIR = Path('data'); DATA_DIR.mkdir(exist_ok=True)
OHLCV_PK = DATA_DIR / 'binance_us_hourly_ohlcv.pk'
CLOSE_PK = DATA_DIR / 'binance_us_hourly_close.pk'
UNIV_CSV = DATA_DIR / 'universe.csv'
FORCE_REFRESH = False        # True = rebuild even if the pickle already exists

In [3]:
# --- 1-2) Candidate universe: coins on BOTH Binance.US and Coinbase, top-N by Binance.US volume ---
def binance_us_bases():
    """{base: (24h_quote_volume, preferred_symbol)} for USD/USDT TRADING pairs, ex-stablecoins."""
    info = client.get_exchange_info()
    bn = {}
    for s in info['symbols']:
        if s['status'] == 'TRADING' and s['quoteAsset'] in ('USDT', 'USD'):
            bn.setdefault(s['baseAsset'], []).append(s['symbol'])
    tick = {t['symbol']: float(t['quoteVolume']) for t in client.get_ticker()}
    out = {}
    for base, syms in bn.items():
        if base in STABLES:
            continue
        vol = max((tick.get(x, 0) for x in syms), default=0)
        sym = next((x for x in syms if x.endswith('USDT')), syms[0])   # prefer the USDT pair
        out[base] = (vol, sym)
    return out

def coinbase_bases():
    """Set of base currencies with an online USD/USDT product on Coinbase."""
    req = urllib.request.Request('https://api.exchange.coinbase.com/products',
                                 headers={'User-Agent': 'research'})
    prods = json.load(urllib.request.urlopen(req, context=SSL_CTX))   # SSL_CTX: macOS cert fix
    return {p['base_currency'] for p in prods
            if p.get('quote_currency') in ('USD', 'USDT')
            and not p.get('trading_disabled') and p.get('status') == 'online'}

base_vol = binance_us_bases()
cb = coinbase_bases()
both = sorted((b for b in base_vol if b in cb), key=lambda b: base_vol[b][0], reverse=True)
candidates = [base_vol[b][1] for b in both[:TOP_N]]
print(f'Binance.US bases={len(base_vol)} | Coinbase bases={len(cb)} | on both={len(both)} | candidates={len(candidates)}')

Binance.US bases=200 | Coinbase bases=401 | on both=139 | candidates=139


In [4]:
# --- 3) Download hourly OHLCV 2022->now, pruning coins listed too late to hit 90% ---
def fetch_ohlcv(symbol, interval=INTERVAL, start=START):
    raw = client.get_historical_klines(symbol, interval, start)
    cols = ['open_time','open','high','low','close','volume','close_time',
            'quote_volume','num_trades','taker_base_volume','taker_quote_volume','ignore']
    df = pd.DataFrame(raw, columns=cols)
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    df = df.set_index('open_time')[['open','high','low','close','volume']].astype(float)
    df.columns = ['Open','High','Low','Close','Volume']
    return df

if OHLCV_PK.exists() and not FORCE_REFRESH:
    panel = pd.read_pickle(OHLCV_PK)
    print('Loaded cached panel (set FORCE_REFRESH=True to rebuild). Skipping steps 3-6.')
else:
    # prune late listings with a cheap 1-bar probe
    survivors = []
    for sym in candidates:
        k = client.get_historical_klines(sym, INTERVAL, START, limit=1)
        if k and pd.to_datetime(k[0][0], unit='ms', utc=True) <= PRUNE_AFTER:
            survivors.append(sym)
        time.sleep(0.05)
    print(f'after listing-date prune: {len(survivors)} / {len(candidates)}')

    frames = {}
    for i, sym in enumerate(survivors, 1):
        t0 = time.time()
        try:
            frames[sym] = fetch_ohlcv(sym)
            print(f'[{i}/{len(survivors)}] {sym}: {len(frames[sym])} bars ({time.time()-t0:.1f}s)')
        except Exception as e:
            print(f'[{i}/{len(survivors)}] {sym} FAIL {str(e)[:60]}')
        time.sleep(0.2)
    panel = pd.concat(frames, axis=1).swaplevel(axis=1).sort_index(axis=1)
    panel = panel.reindex(columns=['Open','High','Low','Close','Volume'], level=0)
    panel.columns.names = ['Field', 'Ticker']

Loaded cached panel (set FORCE_REFRESH=True to rebuild). Skipping steps 3-6.


In [5]:
# --- 4) Reindex to a full hourly UTC grid, drop coins with < MIN_COVERAGE, save ---
if not (OHLCV_PK.exists() and not FORCE_REFRESH):
    full_idx = pd.date_range(pd.Timestamp(START, tz='UTC'), panel.index.max(), freq='1h')
    panel = panel.reindex(full_idx)
    panel.index.name = 'open_time_utc'

    cov = panel['Close'].notna().mean()
    keep = cov[cov >= MIN_COVERAGE].index.tolist()
    dropped = sorted(set(panel['Close'].columns) - set(keep))
    panel = panel.loc[:, (slice(None), keep)]
    print(f'coverage>=90%: kept {len(keep)}  |  dropped {len(dropped)}: {dropped}')

    panel.to_pickle(OHLCV_PK)
    panel['Close'].to_pickle(CLOSE_PK)
    pd.Series(sorted(keep), name='symbol').to_csv(UNIV_CSV, index=False)
    print(f'Saved {OHLCV_PK.name}, {CLOSE_PK.name}, {UNIV_CSV.name}')

print(panel.shape, '|', panel.index.min(), '->', panel.index.max())

(26334, 90) | 2023-08-27 00:00:00+00:00 -> 2026-08-28 05:00:00+00:00


In [6]:
# --- Sanity checks ---
tickers = list(panel.columns.get_level_values('Ticker').unique())
print(f'{len(tickers)} coins:', tickers)
cov = panel['Close'].notna().mean().sort_values()
print('\nlowest coverage:', {k: round(v, 3) for k, v in cov.head(5).items()})
panel['Close'].tail()

18 coins: ['ADAUSDT', 'ATOMUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETHUSDT', 'ICPUSDT', 'LINKUSDT', 'LTCUSDT', 'NEARUSDT', 'SHIBUSDT', 'SOLUSDT', 'UNIUSDT', 'XLMUSDT', 'XRPUSDT']

lowest coverage: {'ADAUSDT': 1.0, 'UNIUSDT': 1.0, 'SOLUSDT': 1.0, 'SHIBUSDT': 1.0, 'NEARUSDT': 1.0}


Ticker,ADAUSDT,ATOMUSDT,AVAXUSDT,BCHUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,DOTUSDT,ETHUSDT,ICPUSDT,LINKUSDT,LTCUSDT,NEARUSDT,SHIBUSDT,SOLUSDT,UNIUSDT,XLMUSDT,XRPUSDT
open_time_utc,,,,,,,,,,,,,,,,,,
2026-08-28 01:00:00+00:00,0.21434,1.581,7.512,269.3,714.49,80493.03,0.08911,0.881,2510.89,2.476,11.895,50.13,1.954,0.000005,107.88,4.737,0.18658,1.4491
2026-08-28 02:00:00+00:00,0.20997,1.568,7.417,264.0,708.19,79936.13,0.08791,0.881,2492.60,2.425,11.718,49.33,1.877,0.000005,106.58,4.577,0.18328,1.4332
2026-08-28 03:00:00+00:00,0.21021,1.536,7.446,264.0,709.22,79829.70,0.08781,0.872,2490.41,2.425,11.694,49.38,1.875,0.000005,106.82,4.664,0.18342,1.4316
2026-08-28 04:00:00+00:00,0.21006,1.511,7.453,264.0,709.64,79642.24,0.08759,0.873,2485.76,2.426,11.735,49.28,1.890,0.000005,106.72,4.655,0.18477,1.4210
2026-08-28 05:00:00+00:00,0.21006,1.519,7.432,264.0,709.79,79625.21,0.08741,0.868,2484.45,2.426,11.729,49.03,1.871,0.000005,106.84,4.605,0.18477,1.4193
